# GH05T3 Fine-Tune — 2x T4 Multi-GPU

**Required Kaggle Settings (right sidebar):**
- Accelerator: **GPU T4 x2** ← critical
- Internet: **On**
- Persistence: Files only

**Data setup (one-time, no HF repo needed):**
1. Go to kaggle.com/datasets → New Dataset
2. Upload these 4 files from `backend/training/datasets/`:
   - `adversarial_defense.jsonl`
   - `reasoning_chains.jsonl`
   - `cve_patterns.jsonl`
   - `bug_bounty.jsonl`
3. Name the dataset **gh05t3-datasets** (or anything — just match the path in Cell 3)
4. Add it as input to this notebook: + Add Data → Your Datasets → gh05t3-datasets

In [ ]:
# Cell 1: Install deps
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

!pip install --quiet --upgrade --no-cache-dir \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    xformers trl peft accelerate bitsandbytes

print('deps installed')

In [ ]:
# Cell 2: Verify GPUs
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | {mem:.1f} GB')
assert torch.cuda.device_count() >= 1, 'No GPU found'
print('GPU check passed')

In [ ]:
# Cell 3: Load data from local Kaggle dataset (no HF repo needed)
import json
import random
from pathlib import Path
from datasets import Dataset

# ── Where Kaggle mounts input datasets ─────────────────────────────────────
# If you named your Kaggle dataset "gh05t3-datasets", it lands at:
DATA_DIR = Path('/kaggle/input/gh05t3-datasets')
# If you used a different name, change the folder name above to match.

SYSTEM_PROMPT = (
    'You are GH05T3, an autonomous security and reasoning agent. '
    'You think carefully, reason step-by-step, and always prioritize '
    'detection and defense over exploitation.'
)

def iter_jsonl(path):
    if not path.exists():
        print(f'  WARNING: {path} not found — skipping')
        return
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    pass

def to_chatml(messages):
    return '\n'.join(
        f'<|im_start|>{m["role"]}\n{m["content"]}<|im_end|>'
        for m in messages
    )

texts = []

# adversarial_defense.jsonl
for rec in iter_jsonl(DATA_DIR / 'adversarial_defense.jsonl'):
    threat = rec.get('threat_vector', '')
    if not threat:
        continue
    resp = (
        f"**Exploitation method:** {rec.get('exploitation_method', 'N/A')}\n\n"
        f"**Detection patterns:** {rec.get('detection_pattern', 'N/A')}\n\n"
        f"**Mitigation strategy:** {rec.get('mitigation_strategy', 'N/A')}"
    )
    texts.append(to_chatml([
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': f'Analyze this threat and provide detection and mitigation guidance:\n\n{threat}'},
        {'role': 'assistant', 'content': resp},
    ]))

# reasoning_chains.jsonl
for rec in iter_jsonl(DATA_DIR / 'reasoning_chains.jsonl'):
    question = rec.get('question', '')
    steps    = rec.get('reasoning_steps', [])
    if not question or not isinstance(steps, list):
        continue
    steps_text = '\n'.join(f'{i+1}. {s}' for i, s in enumerate(steps))
    resp = (
        f"**Reasoning:**\n{steps_text}\n\n"
        f"**Synthesis:** {rec.get('synthesis', 'N/A')}\n\n"
        f"**Answer:** {rec.get('final_answer', 'N/A')}"
    )
    texts.append(to_chatml([
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': question},
        {'role': 'assistant', 'content': resp},
    ]))

# cve_patterns.jsonl
for rec in iter_jsonl(DATA_DIR / 'cve_patterns.jsonl'):
    pattern = rec.get('vulnerability_pattern', '')
    cve_id  = rec.get('source_cve', 'Unknown CVE')
    if not pattern:
        continue
    indicators = rec.get('discovery_indicators', [])
    ind_text   = ('\n'.join(f'• {i}' for i in indicators)
                  if isinstance(indicators, list) else str(indicators))
    resp = (
        f"**Pattern:** {pattern}\n\n"
        f"**Discovery indicators:**\n{ind_text}\n\n"
        f"**Exploitation timeline:** {rec.get('exploitation_timeline', 'N/A')}\n\n"
        f"**Defensive lessons:** {rec.get('defensive_lessons', 'N/A')}"
    )
    texts.append(to_chatml([
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': f'Analyze the vulnerability pattern for {cve_id} and explain detection and defense.'},
        {'role': 'assistant', 'content': resp},
    ]))

# bug_bounty.jsonl
for rec in iter_jsonl(DATA_DIR / 'bug_bounty.jsonl'):
    target = rec.get('target_system', '')
    vuln   = rec.get('vulnerability_found', '')
    if not target or not vuln:
        continue
    resp = (
        f"**Recon method:** {rec.get('recon_method', 'N/A')}\n\n"
        f"**Vulnerability:** {vuln}\n\n"
        f"**Non-weaponized PoC:** {rec.get('non_weaponized_poc', 'N/A')}\n\n"
        f"**Impact:** {rec.get('impact_assessment', 'N/A')}\n\n"
        f"**Remediation:** {rec.get('remediation', 'N/A')}"
    )
    texts.append(to_chatml([
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': f'As a security researcher testing a {target}, describe how to responsibly discover and report a {vuln}.'},
        {'role': 'assistant', 'content': resp},
    ]))

assert texts, (
    f'No training data found at {DATA_DIR}.\n'
    'Did you add your Kaggle dataset as an input? '
    'See the markdown cell at the top for setup steps.'
)

random.seed(42)
random.shuffle(texts)

dataset = Dataset.from_dict({'text': texts})
print(f'Dataset: {len(dataset)} examples')
print('Sample preview:')
print(dataset[0]['text'][:300], '...')

In [ ]:
# Cell 4: Load model
from unsloth import FastLanguageModel

MODEL_ID       = 'unsloth/Qwen2.5-Coder-7B-Instruct'
MAX_SEQ_LENGTH = 512
LORA_RANK      = 32

# unsloth loads to cuda:0 by default; training DDP distributes across both GPUs
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_ID,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = True,
)

print('Model loaded')

In [ ]:
# Cell 5: Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r                       = LORA_RANK,
    target_modules          = ['q_proj','k_proj','v_proj','o_proj',
                               'gate_proj','up_proj','down_proj'],
    lora_alpha              = LORA_RANK,
    lora_dropout            = 0,
    bias                    = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state            = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 6: Train
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = dataset,
    dataset_text_field= 'text',
    max_seq_length    = MAX_SEQ_LENGTH,
    dataset_num_proc  = 2,
    packing           = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,    # 4 per GPU × 2 GPUs = 8
        gradient_accumulation_steps = 2,    # effective batch = 16
        warmup_steps                = 50,
        max_steps                   = 800,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        optim                       = 'adamw_8bit',
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        output_dir                  = 'outputs',
        save_strategy               = 'steps',
        save_steps                  = 200,
        save_total_limit            = 2,
        report_to                   = 'none',
    ),
)

stats = trainer.train()
print(f'Done — loss: {stats.training_loss:.4f}, steps: {stats.global_step}')

In [ ]:
# Cell 7: Save adapter
OUT = 'gh05t3_lora_adapter'
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

import json
with open(f'{OUT}/training_config.json', 'w') as f:
    json.dump({
        'model':        MODEL_ID,
        'lora_rank':    LORA_RANK,
        'max_seq_len':  MAX_SEQ_LENGTH,
        'steps':        stats.global_step,
        'final_loss':   stats.training_loss,
        'dataset_size': len(dataset),
    }, f, indent=2)

print(f'Adapter saved to /kaggle/working/{OUT}')

In [ ]:
# Cell 8: Test inference
FastLanguageModel.for_inference(model)

prompt = '<|im_start|>system\nYou are GH05T3.<|im_end|>\n<|im_start|>user\nExplain SQL injection detection patterns.<|im_end|>\n<|im_start|>assistant\n'
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
out    = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))